In [1]:
import pandas as pd
import subprocess
import numpy as np
from Bio.Align.Applications import MafftCommandline
from Bio import SeqIO
from Bio import AlignIO
from io import StringIO
import os, subprocess
import seaborn as sns
import re

/home/tiago/anaconda3/envs/databases/lib/python3.10/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [2]:
import sys
sys.path.append('../')
from source import Process
import pandas as pd
import json

In [3]:
NcrdClusters = Process.ClusterCounter("../data/filtered/ncrd95.21032025.tagged.fasta")
ResfinderClusters = Process.ClusterCounter("../data/filtered/resfinder.21032025.tagged.fasta")
CardClusters = Process.ClusterCounter("../data/filtered/card.21032025.tagged.fasta")
MegaresClusters = Process.ClusterCounter("../data/filtered/megares.21032025.tagged.fasta")
HmdClusters = Process.ClusterCounter("../data/filtered/hmd.21032025.tagged.fasta")
NdaroClusters = Process.ClusterCounter("../data/filtered/ndaro.21032025.tagged.fasta")

CalledProcessError: Command '['grep', '-c', '>', '../data/filtered/ncrd95.21032025.tagged.fasta']' returned non-zero exit status 2.

In [ ]:
ClusterLimits = pd.DataFrame({
    "Database": ["NCRD", "Resfinder", "CARD", "MEGARES", "HMD", "NDARO"],
    "100%": [NcrdClusters[0][0], ResfinderClusters[0][0], CardClusters[0][0], MegaresClusters[0][0], HmdClusters[0][0], NdaroClusters[0][0]],
    "95%":  [NcrdClusters[0][1], ResfinderClusters[0][1], CardClusters[0][1], MegaresClusters[0][1], HmdClusters[0][1], NdaroClusters[0][1]],
    "90%":  [NcrdClusters[0][2], ResfinderClusters[0][2], CardClusters[0][2], MegaresClusters[0][2], HmdClusters[0][2], NdaroClusters[0][2]],
    "85%":  [NcrdClusters[0][3], ResfinderClusters[0][3], CardClusters[0][3], MegaresClusters[0][3], HmdClusters[0][3], NdaroClusters[0][3]],
    "80%":  [NcrdClusters[0][4], ResfinderClusters[0][4], CardClusters[0][4], MegaresClusters[0][4], HmdClusters[0][4], NdaroClusters[0][4]],  
    "75%":  [NcrdClusters[0][5], ResfinderClusters[0][5], CardClusters[0][5], MegaresClusters[0][5], HmdClusters[0][5], NdaroClusters[0][5]],
    "70%":  [NcrdClusters[0][6], ResfinderClusters[0][6], CardClusters[0][6], MegaresClusters[0][6], HmdClusters[0][6], NdaroClusters[0][6]]
})
ClusterLimits = ClusterLimits.set_index("Database").T
ClusterLimits.columns.name = None
ClusterLimits.to_csv("../data/processed/cdhitclusters.defaultsettings.csv", index=True, sep = "\t")

In [ ]:
PairWiseAlignment = pd.read_csv(
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq1.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)

PairWiseAlignment["qseqtag"] = PairWiseAlignment["qseqid"].str.split("|").str[1]
PairWiseAlignment["sseqtag"] = PairWiseAlignment["sseqid"].str.split("|").str[1]
PairWiseAlignment["qseqid"] = PairWiseAlignment["qseqid"].str.split("|").str[0]
PairWiseAlignment["sseqid"] = PairWiseAlignment["sseqid"].str.split("|").str[0]
PairWiseAlignment = PairWiseAlignment.loc[PairWiseAlignment.pident >= 95]
PairWiseAlignment.replace({"RESFINDER":"ResFinder", "MEGARES":"MegaRes","HMDARG":"HMDARG-DB"}, inplace = True)
BlastPairWiseAlignmentPivoted = PairWiseAlignment.pivot_table(index="qseqtag", columns="sseqtag", values="pident", aggfunc="count").fillna(0)
BlastPairWiseAlignmentPivoted.index.name = None
BlastPairWiseAlignmentPivoted.columns.name = None
BlastPairWiseAlignmentPivoted.to_csv("../data/processed/BlastPairWiseAlignmentPivoted.cov80.maxseq1.csv", index=True, sep = "\t")


In [ ]:
BlastPairWiseAlignmentPivoted

,CARD,HMD,MegaRes,NCRD,NDARO,ResFinder
CARD,25.0,2866.0,1774.0,84.0,1285.0,0.0
HMD,4018.0,6921.0,1283.0,1988.0,264.0,6.0
MegaRes,4786.0,1373.0,164.0,67.0,241.0,196.0
NCRD,2038.0,3564.0,231.0,4963.0,65.0,1.0
NDARO,5771.0,985.0,302.0,40.0,69.0,15.0
ResFinder,2254.0,327.0,336.0,0.0,18.0,10.0


In [ ]:
PairWiseAlignment

,qseqid,sseqid,pident,length,qlen,slen,qstart,qend,sstart,send,evalue,bitscore,ppos,full_qseq,full_sseq,qseqtag,sseqtag
0,CARD_0,MEGARES_1086,100.0,296,296,296,1,296,1,296,5.450000e-215,587.0,100.0,MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAV...,MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAV...,CARD,MegaRes
1,CARD_1,HMD_829,100.0,286,286,286,1,286,1,286,3.770000e-201,551.0,100.0,MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMD...,MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMD...,CARD,HMD
2,CARD_2,NCRD_0,100.0,164,164,164,1,164,1,164,3.050000e-117,328.0,100.0,MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKS...,MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKS...,CARD,NCRD
3,CARD_3,HMD_1374,100.0,291,291,291,1,291,1,291,5.800000e-203,556.0,100.0,MVTKRVQRMMFAAAACIPLLLGSAPLYAQTSAVQQKLAALEKSSGG...,MVTKRVQRMMFAAAACIPLLLGSAPLYAQTSAVQQKLAALEKSSGG...,CARD,HMD
4,CARD_4,HMD_247,100.0,270,270,270,1,270,1,270,2.760000e-195,535.0,100.0,MELPNIMHPVAKLSTALAAALMLSGCMPGEIRPTIGQQMETGDQRF...,MELPNIMHPVAKLSTALAAALMLSGCMPGEIRPTIGQQMETGDQRF...,CARD,HMD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74155,RESFINDER_2950,HMD_5549,100.0,283,283,283,1,283,1,283,1.220000e-200,549.0,100.0,MLRSRVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVV...,MLRSRVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVV...,ResFinder,HMD
74156,RESFINDER_2951,MEGARES_4771,100.0,275,275,275,1,275,1,275,2.870000e-195,535.0,100.0,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,ResFinder,MegaRes
74157,RESFINDER_2952,HMD_8396,100.0,279,279,279,1,279,1,279,7.040000e-198,542.0,100.0,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,ResFinder,HMD
74158,RESFINDER_2953,MEGARES_4770,100.0,279,279,279,1,279,1,279,1.730000e-198,543.0,100.0,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,ResFinder,MegaRes


In [ ]:
CardIndex = pd.read_csv(
    "../data/raw/CARD_Index.tsv",
    sep = "\t"
)
CardDict = CardIndex.drop_duplicates(subset="ARO Accession", keep='first').set_index("ARO Accession").to_dict(orient="index")
## Duplicates
CardIndex.value_counts("ARO Accession")[:10]

ARO Accession
ARO:3002118    2
ARO:3003900    2
ARO:3003893    2
ARO:3000506    2
ARO:3003170    2
ARO:3006379    1
ARO:3006378    1
ARO:3006377    1
ARO:3006376    1
ARO:3006375    1
Name: count, dtype: int64

In [ ]:
NDAROIndex = pd.read_csv(
    "../data/raw/NDARO_Index.tsv", 
    sep="\t")
NdaroDict = NDAROIndex.dropna(subset=["RefSeq protein"]).drop_duplicates(subset="RefSeq protein", keep='first').set_index("RefSeq protein").to_dict(orient="index")
NDAROIndex.value_counts("RefSeq protein")[:10]

RefSeq protein
WP_000090315.1    61
WP_000918664.1    44
WP_003112576.1    32
WP_001281271.1    24
WP_003436174.1    24
WP_001281881.1    23
WP_001281243.1    23
WP_001212189.1    23
WP_003094139.1    22
WP_005693446.1    22
Name: count, dtype: int64

In [ ]:
SCHEMAS = {
    "CARD": {
        "AROSplitPoint": 3,
        "IndexInfo": CardDict
    },
    "NDARO": {
        "AccSplitPoint": 0,
        "IndexInfo": NdaroDict
    },
    "MEGARES": {
        "DrugClassSplitPoint": 2,
        "NameSplitPoint": -1
    },
    "HMD": {
        "DrugClassSplitPoint": 3,
        "NameSplitPoint": 4
    },
    "NCRD": {
        "DrugClassSplitPoint": 4,
        "NameSplitPoint": 2
    },
    "RESFINDER": {
        "DrugClassSplitPoint": 2,
        "NameSplitPoint": 1
    }
}

In [ ]:
MetaDict = Process.CreateMetadataFile(
    "../data/filtered/AllDatabases.21032025.tagged.fasta_ids",
    "../data/processed/MetadataDict.json",
    SCHEMAS
    )
